# VAR Model Tutorial

This notebook demonstrates how to fit a **Vector Autoregression (VAR)** model
to a multivariate time series.

## What is VAR?

A VAR(p) model generalises the univariate AR model to *multiple* time series.
Each variable is modelled as a linear function of its own past values **and**
the past values of all other variables in the system:

$$\mathbf{Y}_t = \mathbf{c} + \sum_{i=1}^{p} \mathbf{A}_i \mathbf{Y}_{t-i} + \mathbf{u}_t$$

where $\mathbf{Y}_t$ is a vector of $K$ variables and $\mathbf{A}_i$ are
$K \times K$ coefficient matrices.

VAR is widely used in macroeconomics and finance for analysing the dynamic
relationships between multiple time series.

## Dataset

We use the **macrodata** dataset bundled with `statsmodels`, which contains
quarterly US macroeconomic data (real GDP, CPI, unemployment, etc.).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from timeseries import moving_average, difference, autocorrelation

## 1. Load and visualise the data

In [ ]:
import statsmodels.api as sm

macro = sm.datasets.macrodata.load_pandas().data

# Select three key macro variables
columns = ["realgdp", "realcons", "realinv"]
df = macro[columns].copy()
df.index = pd.period_range(start="1959Q1", periods=len(df), freq="Q")

df.plot(subplots=True, figsize=(12, 8), title=[c for c in columns])
plt.suptitle("US Macroeconomic Variables (Quarterly)", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print(df.head())

## 2. Explore stationarity with library helpers

VAR requires stationary series, so we difference each variable and check
autocorrelation.

In [ ]:
# Apply log differencing (≈ percentage change) using our library function
df_log = np.log(df)
diff_data = {}
for col in columns:
    diff_data[col] = difference(df_log[col].values.tolist(), lag=1)

# Build a DataFrame of log-differenced series
df_diff = pd.DataFrame(diff_data, index=df.index[1:])

df_diff.plot(subplots=True, figsize=(12, 8),
             title=[f"Δlog({c})" for c in columns])
plt.suptitle("Log-Differenced Series", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Autocorrelation of each differenced series
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, columns):
    vals = diff_data[col]
    acf_vals = [autocorrelation(vals, lag) for lag in range(1, 13)]
    ax.bar(range(1, 13), acf_vals)
    ax.set_title(f"ACF of Δlog({col})")
    ax.set_xlabel("Lag")
plt.tight_layout()
plt.show()

## 3. Lag order selection

In [ ]:
from statsmodels.tsa.api import VAR

var_model = VAR(df_diff.astype(float))
lag_selection = var_model.select_order(maxlags=8)
print(lag_selection.summary())

## 4. Fit the VAR model

In [ ]:
# Use the AIC-optimal lag
optimal_lag = lag_selection.aic
print(f"Optimal lag (AIC): {optimal_lag}")

result = var_model.fit(optimal_lag)
print(result.summary())

## 5. Granger causality tests

Test whether each variable Granger-causes the others.

In [ ]:
for caused in columns:
    for causing in columns:
        if caused != causing:
            test_result = result.test_causality(caused, [causing], kind="f")
            pval = test_result.pvalue
            sig = "*" if pval < 0.05 else ""
            print(f"{causing:>10s} → {caused:<10s}  p={pval:.4f} {sig}")

## 6. Impulse response functions

In [ ]:
irf = result.irf(periods=20)
irf.plot(figsize=(12, 10))
plt.suptitle("Impulse Response Functions", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 7. Forecasting

In [ ]:
n_forecast = 8  # 8 quarters = 2 years
forecast = result.forecast(df_diff.values[-optimal_lag:], steps=n_forecast)
fc_df = pd.DataFrame(forecast, columns=columns,
                      index=pd.period_range(start=df_diff.index[-1] + 1,
                                            periods=n_forecast, freq="Q"))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, columns):
    ax.plot(df_diff[col].iloc[-20:].values, label="Observed")
    fc_vals = fc_df[col].values
    ax.plot(range(20, 20 + n_forecast), fc_vals, marker="o",
            color="red", label="Forecast")
    ax.set_title(f"Δlog({col})")
    ax.legend(fontsize=8)
plt.suptitle("VAR Forecast — 8 Quarters Ahead", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Forecast Error Variance Decomposition (FEVD)

In [ ]:
fevd = result.fevd(periods=20)
fevd.plot(figsize=(12, 8))
plt.suptitle("Forecast Error Variance Decomposition", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## Summary

| Step | Tool / Function |
|---|---|
| Differencing for stationarity | `timeseries.difference` |
| ACF analysis | `timeseries.autocorrelation` |
| Lag selection, VAR fit | `statsmodels.tsa.api.VAR` |
| Granger causality | `VARResults.test_causality` |
| Impulse responses & FEVD | `VARResults.irf`, `VARResults.fevd` |